In [14]:
import pandas as pd
import numpy as np

recipes = pd.read_csv("recipes.csv")
portion_uom_types = pd.read_csv("portion_uom_types.csv")
pos = pd.read_csv("pos_ordersale.csv")
menuitem = pd.read_csv("menuitem.csv")
menu_items = pd.read_csv("menu_items.csv")
ingredients = pd.read_csv("ingredients.csv")
recipe_ingredient_assign = pd.read_csv("recipe_ingredient_assignments.csv")
recipe_sub_recipe = pd.read_csv("recipe_sub_recipe_assignments.csv")
sub_recipe = pd.read_csv("sub_recipes.csv")
sub_recipe_ing = pd.read_csv("sub_recipe_ingr_assignments.csv")

In [15]:
merge1 = pd.merge(pos[["MD5KEY_ORDERSALE", "StoreNumber", "date"]],
                  menuitem[["MD5KEY_ORDERSALE", "MD5KEY_MENUITEM", "Quantity", "Id"]],
                  how="inner", on="MD5KEY_ORDERSALE").rename(columns={'Quantity': 'order_quantity'})

merge2 = pd.merge(merge1, menu_items[["RecipeId", "MenuItemId"]], how='left', left_on='Id', right_on='MenuItemId')
merge3 = pd.merge(merge2, recipe_ingredient_assign, how='left', on='RecipeId')
merge4 = pd.merge(merge3, ingredients, how='inner', on='IngredientId')
merge5 = pd.merge(merge4, portion_uom_types, how='inner', on='PortionUOMTypeId')

merge5['Store_name'] = merge5.StoreNumber.map({46673: 'California 1 (ID:46673)', 
                                               4904: 'California 2 (ID:4904)', 
                                               12631: 'New York 1 (ID:12631)', 
                                               20974: 'New York 2 (ID:20974)'})

merge2_sub = pd.merge(merge2, recipe_sub_recipe, how='left', on='RecipeId')
merge3_sub = pd.merge(merge2_sub, sub_recipe_ing, how='left', on='SubRecipeId')
merge4_sub = pd.merge(merge3_sub, ingredients, how='inner', on='IngredientId')
merge5_sub = pd.merge(merge4_sub, portion_uom_types, how='inner', on='PortionUOMTypeId')

merge5_sub['Store_name'] = merge5_sub.StoreNumber.map({46673: 'California 1 (ID:46673)', 
                                                       4904: 'California 2 (ID:4904)', 
                                                       12631: 'New York 1 (ID:12631)', 
                                                       20974: 'New York 2 (ID:20974)'})

In [16]:
Lettuce_main = merge5[merge5.IngredientShortDescription.str.contains('Lettuce')]
Lettuce_sub = merge5_sub[merge5_sub.IngredientShortDescription.str.contains('Lettuce')]

Lettuce_main['ttl_quantity'] = Lettuce_main['order_quantity'] * Lettuce_main['Quantity']
Lettuce_sub['ttl_quantity'] = Lettuce_sub['order_quantity'] * Lettuce_sub['Quantity'] * Lettuce_sub['Factor']

Lettuce_main_agg = Lettuce_main.groupby(['Store_name', 'date']).agg({'ttl_quantity': 'sum'}).reset_index()
Lettuce_sub_agg = Lettuce_sub.groupby(['Store_name', 'date']).agg({'ttl_quantity': 'sum'}).reset_index()

lettuce_main_pivot = Lettuce_main_agg.pivot(index='date', columns='Store_name', values='ttl_quantity').fillna(0)
lettuce_sub_pivot = Lettuce_sub_agg.pivot(index='date', columns='Store_name', values='ttl_quantity').fillna(0)

lettuce_main_pivot
lettuce_sub_pivot
lettuce_combined = pd.merge(lettuce_main_pivot, lettuce_sub_pivot, how='outer', on='date', suffixes=('_main', '_sub'))

/var/folders/m5/r0nkt_nx14bcd4qdrh71m_0m0000gn/T/ipykernel_11287/1049899032.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Lettuce_main['ttl_quantity'] = Lettuce_main['order_quantity'] * Lettuce_main['Quantity']
/var/folders/m5/r0nkt_nx14bcd4qdrh71m_0m0000gn/T/ipykernel_11287/1049899032.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Lettuce_sub['ttl_quantity'] = Lettuce_sub['order_quantity'] * Lettuce_sub['Quantity'] * Lettuce_sub['Factor']


In [17]:
lettuce_combined.fillna(0, inplace=True)

In [18]:
lettuce_combined['store_46673 (ounces)'] = lettuce_combined['California 1 (ID:46673)_main'] + lettuce_combined['California 1 (ID:46673)_sub']
lettuce_combined['store_4904 (ounces)'] = lettuce_combined['California 2 (ID:4904)_main'] + lettuce_combined['California 2 (ID:4904)_sub']
lettuce_combined['store_12631 (ounces)'] = lettuce_combined['New York 1 (ID:12631)_main'] + lettuce_combined['New York 1 (ID:12631)_sub']
lettuce_combined['store_20974 (ounces)'] = lettuce_combined['New York 2 (ID:20974)_main'] + lettuce_combined['New York 2 (ID:20974)_sub']

In [19]:
lettuce_combined

Store_name,California 1 (ID:46673)_main,California 2 (ID:4904)_main,New York 1 (ID:12631)_main,New York 2 (ID:20974)_main,California 1 (ID:46673)_sub,California 2 (ID:4904)_sub,New York 1 (ID:12631)_sub,New York 2 (ID:20974)_sub,store_46673 (ounces),store_4904 (ounces),store_12631 (ounces),store_20974 (ounces)
date,,,,,,,,,,,,
15-03-05,0.0,0.0,10.0,0.0,152.0,0.0,222.0,0.0,152.0,0.0,232.0,0.0
15-03-06,0.0,0.0,5.0,0.0,100.0,0.0,222.0,4.0,100.0,0.0,227.0,4.0
15-03-07,0.0,0.0,5.0,0.0,54.0,0.0,176.0,0.0,54.0,0.0,181.0,0.0
15-03-08,5.0,0.0,5.0,0.0,194.0,0.0,216.0,0.0,199.0,0.0,221.0,0.0
15-03-09,0.0,0.0,25.0,0.0,166.0,0.0,204.0,0.0,166.0,0.0,229.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
15-06-11,5.0,0.0,20.0,0.0,146.0,256.0,356.0,386.0,151.0,256.0,376.0,386.0
15-06-12,10.0,5.0,20.0,0.0,124.0,218.0,346.0,170.0,134.0,223.0,366.0,170.0
15-06-13,0.0,0.0,25.0,0.0,96.0,216.0,204.0,206.0,96.0,216.0,229.0,206.0


In [20]:
lettuce_combined.drop(columns=[
    'California 1 (ID:46673)_main', 'California 1 (ID:46673)_sub',
    'California 2 (ID:4904)_main', 'California 2 (ID:4904)_sub',
    'New York 1 (ID:12631)_main', 'New York 1 (ID:12631)_sub',
    'New York 2 (ID:20974)_main', 'New York 2 (ID:20974)_sub'
], inplace=True)

In [21]:
lettuce_combined

Store_name,store_46673 (ounces),store_4904 (ounces),store_12631 (ounces),store_20974 (ounces)
date,,,,
15-03-05,152.0,0.0,232.0,0.0
15-03-06,100.0,0.0,227.0,4.0
15-03-07,54.0,0.0,181.0,0.0
15-03-08,199.0,0.0,221.0,0.0
15-03-09,166.0,0.0,229.0,0.0
...,...,...,...,...
15-06-11,151.0,256.0,376.0,386.0
15-06-12,134.0,223.0,366.0,170.0
15-06-13,96.0,216.0,229.0,206.0


In [22]:
lettuce_combined.reset_index(inplace=True)
lettuce_combined['date'] = pd.to_datetime(lettuce_combined['date'], format='%y-%m-%d')
lettuce_combined['date'] = lettuce_combined['date'].dt.strftime('%d/%m/%Y')
lettuce_combined

Store_name,date,store_46673 (ounces),store_4904 (ounces),store_12631 (ounces),store_20974 (ounces)
0,05/03/2015,152.0,0.0,232.0,0.0
1,06/03/2015,100.0,0.0,227.0,4.0
2,07/03/2015,54.0,0.0,181.0,0.0
3,08/03/2015,199.0,0.0,221.0,0.0
4,09/03/2015,166.0,0.0,229.0,0.0
...,...,...,...,...,...
98,11/06/2015,151.0,256.0,376.0,386.0
99,12/06/2015,134.0,223.0,366.0,170.0
100,13/06/2015,96.0,216.0,229.0,206.0
101,14/06/2015,138.0,350.0,234.0,167.0


In [23]:
lettuce_combined.to_csv("lettuce_cleaned.csv")